# 2.11 课程学习 (Curriculum Learning)

模仿人类学习过程，按从易到难的顺序组织训练，提升学习效率和最终性能。

本节涵盖：
- 预定义课程
- 自动课程
- 难度调度
- Teacher-forcing课程

## 1. 预定义课程

**目的**：按先验知识安排训练顺序

**基本原理**：根据领域知识预先定义数据从易到难的顺序，如先训练短文本再训练长文本，先训练通用领域再训练专业领域。

**常见策略**：
- 按文本长度：短→长
- 按领域通用性：通用→专业
- 按任务复杂度：简单任务→复杂任务
- 按数据质量：高质量→全部数据

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)

class SimpleClassifier(nn.Module):
    def __init__(self, input_dim=20, hidden=32, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(), nn.Linear(hidden, n_classes))
    def forward(self, x):
        return self.net(x)

n_samples = 300
X_easy = torch.randn(100, 20) * 0.5 + torch.tensor([1.0, -1.0] * 10)
y_easy = (X_easy[:, 0] > 0).long()
X_medium = torch.randn(100, 20) + torch.tensor([0.5, -0.5] * 10)
y_medium = (X_medium[:, 0] + X_medium[:, 1] > 0).long()
X_hard = torch.randn(100, 20)
y_hard = (X_hard[:, :3].sum(dim=1) > 0).long()

model_curriculum = SimpleClassifier()
model_random = SimpleClassifier()
model_random.load_state_dict(model_curriculum.state_dict())

opt_c = torch.optim.Adam(model_curriculum.parameters(), lr=1e-3)
opt_r = torch.optim.Adam(model_random.parameters(), lr=1e-3)

print('=== Predefined Curriculum vs Random Training ===')
for phase in range(3):
    data = [(X_easy, y_easy), (X_medium, y_medium), (X_hard, y_hard)][:phase+1]
    X_curr = torch.cat([d[0] for d in data])
    y_curr = torch.cat([d[1] for d in data])
    for epoch in range(10):
        logits = model_curriculum(X_curr)
        loss = F.cross_entropy(logits, y_curr)
        opt_c.zero_grad()
        loss.backward()
        opt_c.step()

X_all = torch.cat([X_easy, X_medium, X_hard])
y_all = torch.cat([y_easy, y_medium, y_hard])
for epoch in range(30):
    perm = torch.randperm(len(X_all))
    logits = model_random(X_all[perm])
    loss = F.cross_entropy(logits, y_all[perm])
    opt_r.zero_grad()
    loss.backward()
    opt_r.step()

with torch.no_grad():
    acc_c = (model_curriculum(X_all).argmax(-1) == y_all).float().mean()
    acc_r = (model_random(X_all).argmax(-1) == y_all).float().mean()
print(f'Curriculum training accuracy: {acc_c:.3f}')
print(f'Random training accuracy:     {acc_r:.3f}')
print('\nKey: Curriculum learning starts with easy data, gradually adds harder data.')

## 2. 自动课程

**目的**：自动评估样本难度并排序

**基本原理**：使用训练中的损失值、预测置信度或独立难度评估模型自动衡量样本难度，按难度递增的顺序送入训练。比预定义课程更灵活，能适应数据特点。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class AutoCurriculumScheduler:
    def __init__(self, model, X, y, n_bins=5):
        self.model = model
        self.n_bins = n_bins
        with torch.no_grad():
            losses = F.cross_entropy(model(X), y, reduction='none')
        sorted_indices = losses.argsort()
        self.bins = torch.array_split(sorted_indices, n_bins)

    def get_batch(self, progress, batch_size=32):
        max_bin = min(int(progress * self.n_bins) + 1, self.n_bins)
        pool = torch.cat(self.bins[:max_bin])
        idx = torch.randperm(len(pool))[:batch_size]
        return pool[idx]

model = nn.Sequential(nn.Linear(20, 32), nn.ReLU(), nn.Linear(32, 3))
X = torch.randn(500, 20)
y = (X[:, :3].sum(dim=1) > 0).long()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = AutoCurriculumScheduler(model, X, y, n_bins=5)

print('=== Auto Curriculum (Loss-based Difficulty) ===')
for bin_id in range(5):
    bin_indices = scheduler.bins[bin_id]
    with torch.no_grad():
        bin_loss = F.cross_entropy(model(X[bin_indices]), y[bin_indices])
    print(f'Bin {bin_id+1} (easiest->hardest): {len(bin_indices)} samples, avg_loss={bin_loss:.3f}')

n_steps = 50
for step in range(n_steps):
    progress = step / n_steps
    indices = scheduler.get_batch(progress, batch_size=64)
    logits = model(X[indices])
    loss = F.cross_entropy(logits, y[indices])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (step+1) % 10 == 0:
        acc = (model(X).argmax(-1) == y).float().mean()
        print(f'Step {step+1} (progress={progress:.0%}): loss={loss.item():.4f}, total_acc={acc:.3f}')

print('\nKey: Auto curriculum sorts samples by loss (easy=low loss, hard=high loss).')

## 3. 难度调度

**目的**：控制训练过程中难度的增长节奏

**基本原理**：类似学习率调度，设计难度增长曲线（线性、余弦等），控制何时引入更难的样本，避免过早引入困难样本导致训练不稳定。

**调度策略**：
- **线性调度**：难度均匀增长
- **余弦调度**：先快后慢
- **阶梯调度**：分阶段跳跃增长
- **Baby Step调度**：每次增加一小批更难的数据

In [ ]:
import torch
import numpy as np

def linear_schedule(progress):
    return progress

def cosine_schedule(progress):
    return 0.5 * (1 - np.cos(progress * np.pi))

def step_schedule(progress, n_steps=5):
    return min(1.0, np.floor(progress * n_steps + 1) / n_steps)

def baby_step_schedule(progress, n_steps=8):
    return min(1.0, (np.floor(progress * n_steps) + 1) / n_steps)

progress_points = np.linspace(0, 1, 100)
schedules = {
    'linear': [linear_schedule(p) for p in progress_points],
    'cosine': [cosine_schedule(p) for p in progress_points],
    'step(5)': [step_schedule(p, 5) for p in progress_points],
    'baby_step(8)': [baby_step_schedule(p, 8) for p in progress_points],
}

print('=== Difficulty Scheduling Strategies ===')
print(f'{"Progress":>10s}', end='')
for name in schedules:
    print(f' {name:>12s}', end='')
print()
print('-' * 60)
for p in [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 1.0]:
    print(f'{p:>10.1f}', end='')
    for name, vals in schedules.items():
        idx = int(p * 99)
        print(f' {vals[idx]:>12.3f}', end='')
    print()

print('\nKey insights:')
print('  Linear: steady difficulty increase')
print('  Cosine: fast start, slow finish (good for fine-grained learning)')
print('  Step: discrete jumps between difficulty levels')
print('  Baby step: gradual addition of harder data batches')

## 4. Teacher-Forcing课程

**目的**：在训练和推理之间逐步过渡

**基本原理**：训练时逐步从完全使用真实前文（teacher forcing）过渡到使用模型自身预测作为前文（scheduled sampling），减少训练-推理的分布差异（exposure bias）。

**Exposure Bias问题**：
- 训练时：模型总是看到正确的前文
- 推理时：模型看到的是自己生成的（可能有错的）前文
- 差异导致错误累积

**Scheduled Sampling**：以概率p使用模型自身预测，p从0逐渐增加到1

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class SimpleLM(nn.Module):
    def __init__(self, vocab_size=20, embed_dim=16, hidden=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden, batch_first=True)
        self.head = nn.Linear(hidden, vocab_size)
    def forward(self, x, hidden=None):
        emb = self.embedding(x)
        out, hidden = self.rnn(emb, hidden)
        return self.head(out), hidden

def scheduled_sampling_train(model, target_seq, max_use_model_prob=0.5, n_steps=30):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    vocab_size = 20
    seq_len = target_seq.size(1)

    for step in range(n_steps):
        model_prob = min(max_use_model_prob, step / n_steps * max_use_model_prob * 2)
        input_seq = target_seq[:, :-1].clone()
        hidden = None
        for t in range(seq_len - 1):
            if torch.rand(1).item() < model_prob and t > 0:
                with torch.no_grad():
                    logits_t, hidden = model(input_seq[:, :t+1], hidden)
                    input_seq[:, t] = logits_t[:, -1].argmax(-1)
                    hidden = hidden.detach()

        logits, _ = model(input_seq)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), target_seq[:, 1:].reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (step+1) % 10 == 0:
            acc = (logits.argmax(-1) == target_seq[:, 1:]).float().mean()
            print(f'  Step {step+1}: model_prob={model_prob:.2f}, loss={loss.item():.4f}, acc={acc:.3f}')

model = SimpleLM()
target_seq = torch.randint(0, 20, (16, 8))

print('=== Scheduled Sampling (Teacher-Forcing Curriculum) ===')
print('Gradually transitioning from teacher forcing to model predictions...')
scheduled_sampling_train(model, target_seq, max_use_model_prob=0.5, n_steps=30)

print('\nKey: model_prob starts at 0 (pure teacher forcing) and increases.')
print('This reduces exposure bias between training and inference.')